In [0]:
from pyspark.sql.functions import *
import requests, json
from urllib.parse import quote

In [0]:
def ibm_jobs(key_words):
    url = "https://www-api.ibm.com/search/api/v2"

    headers = {
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Content-Type": "application/json",
        "Referer": "https://www.ibm.com/",
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/150.0.0.0 Safari/537.36"
        ),
    }
    jobs_l = []
    for key in key_words:
        payload = {
            "appId": "careers",
            "scopes": ["careers2"],
            "query": {
                "bool": {
                    "must": [
                        {
                            "simple_query_string": {
                                "query": key,
                                "fields": [
                                    "keywords^1",
                                    "body^1",
                                    "url^2",
                                    "description^2",
                                    "h1s_content^2",
                                    "title^3",
                                    "field_text_01"
                                ]
                            }
                        }
                    ]
                }
            },
            "post_filter": {
                "term": {
                    "field_keyword_05": "India"
                }
            },
            "aggs": {
                "field_keyword_172": {
                    "filter": {"term": {"field_keyword_05": "India"}},
                    "aggs": {
                        "field_keyword_17": {
                            "terms": {"field": "field_keyword_17", "size": 6}
                        },
                        "field_keyword_17_count": {
                            "cardinality": {"field": "field_keyword_17"}
                        }
                    }
                },
                "field_keyword_083": {
                    "filter": {"term": {"field_keyword_05": "India"}},
                    "aggs": {
                        "field_keyword_08": {
                            "terms": {"field": "field_keyword_08", "size": 6}
                        },
                        "field_keyword_08_count": {
                            "cardinality": {"field": "field_keyword_08"}
                        }
                    }
                },
                "field_keyword_184": {
                    "filter": {"term": {"field_keyword_05": "India"}},
                    "aggs": {
                        "field_keyword_18": {
                            "terms": {"field": "field_keyword_18", "size": 6}
                        },
                        "field_keyword_18_count": {
                            "cardinality": {"field": "field_keyword_18"}
                        }
                    }
                },
                "field_keyword_055": {
                    "filter": {"match_all": {}},
                    "aggs": {
                        "field_keyword_05": {
                            "terms": {"field": "field_keyword_05", "size": 1000}
                        },
                        "field_keyword_05_count": {
                            "cardinality": {"field": "field_keyword_05"}
                        }
                    }
                },
                "no_filters_top_hits": {
                    "top_hits": {
                        "size": 1,
                        "_source": False
                    }
                }
            },
            "size": 30,
            "sort": [
            {"_score": "desc"},
            {"dcdate": "desc"}
            ],
            "lang": "zz",
            "localeSelector": {},
            "sm": {
                "query": key,
                "lang": "zz"
            },
            "_source": [
                    "_id",
                    "title",
                    "url",
                    "description",
                    "language",
                    "entitled",
                    "field_keyword_17",
                    "field_keyword_08",
                    "field_keyword_18",
                    "field_keyword_19"
                ]
        }

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        # print(response.status_code)
        data = response.json()

        if response.ok:
            for job in data['hits']['hits']:
                d = {
                    "id": str(job['_source']['url'].split("=")[1]),
                    "title": job['_source']['title'],
                    "location": job['_source']['field_keyword_19'],
                    "url": job['_source']['url'],
                    "date": job['sort'][1]/1000
                }
                jobs_l.append(d)
        else:
            print(response.text)
    df = spark.createDataFrame(jobs_l)
    return df.select("title", "location", lit("IBM").alias("company"), "url", "id", col("date").cast("timestamp").alias("date"), lit('').alias('contact_email')).distinct()

In [0]:
def jpmc_jobs(key_words):
    url = (
        "https://jpmc.fa.oraclecloud.com/hcmRestApi/resources/latest/"
        "recruitingCEJobRequisitions"
    )

    headers = {
        "Accept": "*/*",
        "Accept-Language": "en",
        "Content-Type": "application/vnd.oracle.adf.resourceitem+json;charset=utf-8",
        "Ora-Irc-Language": "en",
        "Referer": (
            "https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/"
            "CX_1001/jobs?keyword=data+engineer&location=India"
            "&locationId=300000000289360&locationLevel=country&mode=location"
        ),
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/150.0.0.0 Safari/537.36"
        )
    }
    jobs_l = []
    for key in key_words:
        params = {
            "onlyData": "true",
            "expand": "requisitionList.workLocation,requisitionList.otherWorkLocations,"
                    "requisitionList.secondaryLocations,flexFieldsFacet.values,"
                    "requisitionList.requisitionFlexFields",
            "finder": (
                "findReqs;"
                "siteNumber=CX_1001,"
                "facetsList=LOCATIONS;WORK_LOCATIONS;WORKPLACE_TYPES;"
                "TITLES;CATEGORIES;ORGANIZATIONS;POSTING_DATES;FLEX_FIELDS,"
                "limit=25,"
                f"keyword=\"{key}\","
                "locationId=300000000289360,"
                "sortBy=RELEVANCY"
            )
        }

        response = requests.get(
            url,
            params=params,
            headers=headers
        )

        if response.ok:
            data = response.json()
            for job in data['items'][0]['requisitionList']:
                d = {
                    "id": str(job['Id']),
                    "title": job["Title"],
                    "date": job["PostedDate"],
                    "location": job["PrimaryLocation"],
                    "url": "https://jpmc.fa.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_1001/job/" + job["Id"]
                }
                jobs_l.append(d)
        else:
            print(response.text)
    df = spark.createDataFrame(jobs_l)
    return df.select("title", "location", lit("JPMorgan").alias("company"), "url", "id", col("date").cast("timestamp").alias("date"), lit('').alias('contact_email')).distinct()

In [0]:
def kpmg_jobs(key_words):
    url = (
        "https://ejgk.fa.em2.oraclecloud.com/hcmRestApi/resources/latest/"
        "recruitingCEJobRequisitions"
    )
    
    sites = ["CX_1", "CX_3"]
    jobs_l = []
    for site in sites:
        params = {
            "onlyData": "true",
            "expand": (
                "requisitionList.workLocation,"
                "requisitionList.otherWorkLocations,"
                "requisitionList.secondaryLocations,"
                "flexFieldsFacet.values,"
                "requisitionList.requisitionFlexFields"
            ),
            "finder": (
                "findReqs;"
                f"siteNumber={site},"
                "facetsList=LOCATIONS;WORK_LOCATIONS;WORKPLACE_TYPES;"
                "TITLES;CATEGORIES;ORGANIZATIONS;POSTING_DATES;FLEX_FIELDS,"
                "limit=25,"
                "keyword=\"data engineer\","
                "locationId=300000000296042,"
                "sortBy=RELEVANCY"
            )
        }
        headers = {
            "Accept": "*/*",
            "Accept-Language": "en",
            "Content-Type": "application/vnd.oracle.adf.resourceitem+json;charset=utf-8",
            "Ora-Irc-Cx-Userid": "93acdd7f-2a9a-4415-a7b2-c775e481a52a",
            "Ora-Irc-Language": "en",
            "Referer": (
                f"https://ejgk.fa.em2.oraclecloud.com/hcmUI/CandidateExperience/en/sites/{site}/jobs"
                "?keyword=data+engineer"
                "&location=India"
                "&locationId=300000000296042"
                "&locationLevel=country"
                "&mode=location"
            ),
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/150.0.0.0 Safari/537.36"
            ),
        }
        for key in key_words:
            params = {
                "onlyData": "true",
                "expand": (
                    "requisitionList.workLocation,"
                    "requisitionList.otherWorkLocations,"
                    "requisitionList.secondaryLocations,"
                    "flexFieldsFacet.values,"
                    "requisitionList.requisitionFlexFields"
                ),
                "finder": (
                    "findReqs;"
                    "siteNumber=CX_1,"
                    "facetsList=LOCATIONS;WORK_LOCATIONS;WORKPLACE_TYPES;"
                    "TITLES;CATEGORIES;ORGANIZATIONS;POSTING_DATES;FLEX_FIELDS,"
                    "limit=25,"
                    "keyword=\"data engineer\","
                    "locationId=300000000296042,"
                    "sortBy=RELEVANCY"
                )
            }
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=30
            )

            if response.ok:
                data = response.json()
                for job in data['items'][0]['requisitionList']:
                            d = {
                                "id": str(job['Id']),
                                "title": job["Title"],
                                "date": job["PostedDate"],
                                "location": job["PrimaryLocation"],
                                "url": f"https://ejgk.fa.em2.oraclecloud.com/hcmUI/CandidateExperience/en/sites/{site}/job/" + job["Id"]
                            }
                            jobs_l.append(d)
            else:
                print(response.text)
    df = spark.createDataFrame(jobs_l)
    return df.select("title", "location", lit("kpmg").alias("company"), "url", "id", col("date").cast("timestamp").alias("date"), lit('').alias('contact_email')).distinct()

In [0]:
def epam_jobs(key_words):
    url = "https://careers.epam.com/api/jobs/v2/search/careers-i18n"

    headers = {
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/150.0.0.0 Safari/537.36"
        ),
        "Referer": (
            "https://careers.epam.com/en/jobs/india"
            "?search=data%20engineer&sort_by=relevance"
        ),
        "X-Anywhere-Tenant": "anywhere",
        "Traceparent": "00-45ddf229f64d38df4d96b443d6a5e19a-6693247a98c30a15-00"
    }
    jobs_l = []
    for key in key_words:
        params = {
            "facets": "country=4060741400035606931",
            "from": 0,
            "lang": "en",
            "q": key,
            "size": 10,
            "sortBy": "relevance;relocation=asc",
            "websiteLocale": "en-us"
        }
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=30
        )


        if response.ok:
            data = response.json()
            for job in data['data']['jobs']:
                cities = ', '.join([city['name'] for city in job['city']])
                d = {
                    "title": job["name"],
                    "contact_email": ','.join(job["contact_email"]),
                    "location": cities,
                    "date": job['updated_at'],
                    "id": '',
                    "url": "https://careers.epam.com/" + job['seo']['url']
                }
                # print(d)
                jobs_l.append(d)
        else:
            print(response.text)
    df = spark.createDataFrame(jobs_l)
    return df.select("title", "location", lit("EPAM").alias("company"), "url", "id", col("date").cast("timestamp").alias("date"), "contact_email").distinct()

In [0]:
def ltm_jobs(key_words):
    url = "https://ltimindtree.ripplehire.com/candidate/candidatejobsearch"

    headers = {
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Accept-Language": "en-US,en;q=0.9",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/150.0.0.0 Safari/537.36"
        ),
        "X-Requested-With": "XMLHttpRequest",
    }
    jobs_l = []
    for key in key_words:
        payload = {
            "careerSiteUrlParams": json.dumps({
                "page": 0,
                "search": key,
                "token": "xviyQvbnyYZdGtozXoNm",
                "source": "CAREERSITE",
                "pagesize": 10,
                "geo": "India"
            }),
            "lang": "en"
        }

        response = requests.post(
            url,
            headers=headers,
            data=payload,
            timeout=30
        )

        # print(response.status_code)

        if response.ok:
            data = response.json()
            for job in data['jobVoList']:
                d = {
                    "id": str(job['jobId']),
                    "title": job['jobTitle'],
                    "location": job['jobLocation'],
                    "date": '',
                    "url": 'https://ltimindtree.ripplehire.com/candidate/?token=xviyQvbnyYZdGtozXoNm&lang=en&source=CAREERSITE#detail/job/' + job['jobId']
                }
                jobs_l.append(d)
        else:
            print(response.text)
    d = spark.createDataFrame(jobs_l)
    return d.select("title", "location", lit("LTM").alias("company"), "url", "id", lit(None).cast('timestamp').alias('date'), lit('').alias('contact_email')).distinct()

In [0]:
def capgemini_jobs(key_words):
    url = "https://cg-jobstream-api.azurewebsites.net/api/job-search"

    headers = {
        "accept": "application/json, text/plain, */*",
    }
    jobs_l = []
    for key in key_words:
        params = {
            "country_code": "in-en",
            "page": 1,
            "size": 11,
            "search": key,
        }
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=30
        )

        if response.ok:
            data = response.json()
            for job in data['data']:
                # print(job)
                d = {
                    "title": job['title'],
                    "date": job['indexed_at'],
                    "id": str(job['ref'].split('-')[0]),
                    "url": job['apply_job_url'].split('?')[0],
                    "location": job['location']
                }
                jobs_l.append(d)
            # print(json.dumps(data, indent=4))
        else:
            print(response.text)
    df = spark.createDataFrame(jobs_l)
    return df.select("title", "location", lit("Capgemini").alias("company"), "url", "id", col("date").cast("timestamp").alias("date"), lit('').alias('contact_email')).distinct()  

In [0]:
def hcl_jobs(key_words):
    url = "https://careers.hcltech.com/services/recruiting/v1/jobs"

    headers = {
        "Accept": "*/*",
        "Accept-Language": "en-GB,en-US;q=0.9,en;q=0.8,ta;q=0.7",
        "Content-Type": "application/json",
        "Referer": (
            "https://careers.hcltech.com/search/"
            "?q=data+engineer"
            "&locationsearch="
            "&searchResultView=LIST"
            "&markerViewed="
            "&carouselIndex="
            "&facetFilters=%7B%22custCountryRegion%22%3A%5B%22India%22%5D%7D"
            "&pageNumber=0"
        ),
    }
    jobs_l = []
    for key in key_words:
        payload = {
            "locale": "en_US",
            "pageNumber": 0,
            "sortBy": "",
            "keywords": key,
            "location": "",
            "facetFilters": {
                "custCountryRegion": ["India"]
            },
            "brand": "",
            "skills": [],
            "categoryId": 0,
            "alertId": "",
            "rcmCandidateId": ""
        }

        session = requests.Session()

        response = session.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )


        try:
            data = response.json()
            for job in data['jobSearchResult']:
                job = job['response']
                # print("https://careers.hcltech.com/job/" + job['urlTitle'] + "/" + job['id'] + '-en_US')
                d = {
                    "id": str(job['id']),
                    "date": job['unifiedStandardStart'],
                    "title": job['unifiedStandardTitle'],
                    "location": "India",
                    "url": "https://careers.hcltech.com/job/" + job['urlTitle'] + "/" + job['id'] + '-en_US'
                }
                jobs_l.append(d)
        except Exception:
            print(response.text)
    df = spark.createDataFrame(jobs_l)
    return(df.select("title", "location", lit("HCL").alias("company"), "url", "id", to_timestamp("date", "M/d/yy").alias("date"), lit('').alias('contact_email')).distinct()) 

In [0]:
def deloitte_jobs(key_words):
    jobs_l = []
    for keyword in key_words:

        url = (
            "https://usijobs.deloitte.com/en_US/careersUSI/"
            f"SearchJobsAJAXJSON/{quote(keyword)}"
        )

        headers = {
            "Accept": "application/json, text/javascript, */*; q=0.01",
            "Accept-Language": "en-US,en;q=0.9",
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/150.0.0.0 Safari/537.36"
            ),
            "X-Requested-With": "XMLHttpRequest",
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=30
        )

        # print(response.status_code)

        if response.ok:
            data = response.json()
            for job in data:
                # print(job)
                d = {
                    "id": str(job['id']),
                    "title": job['value'],
                    "location": job['location'],
                    "date": job['postedDate'],
                    "url": 'https://usijobs.deloitte.com/en_US/careersUSI/JobDetail?jobId=' + str(job['id'])
                }
                jobs_l.append(d)
        else:
            print(response.text)
    df = spark.createDataFrame(jobs_l)
    return df.select("title", "location", lit("Deloitte").alias("company"), "url", "id", to_timestamp("date", "dd-MMM-yyyy").alias("date"), lit('').alias('contact_email')).distinct()

In [0]:
search_keys = ["Azure", "Data Engineer", "Databricks", "Big Data", "ETL", "Data Platform", "Pyspark", "SQL"]
df = ibm_jobs(search_keys)
df = df.union(jpmc_jobs(search_keys))
df = df.union(kpmg_jobs(search_keys))
df = df.union(epam_jobs(search_keys))
df = df.union(ltm_jobs(search_keys))
df = df.union(capgemini_jobs(search_keys))
df = df.union(hcl_jobs(search_keys))
df = df.union(deloitte_jobs(search_keys))
display(df)

In [0]:
df.write.mode("overwrite").saveAsTable("raw_catalogue.jobs.career_portals")